In [1]:
!pip install requests pandas faker

import requests
import pandas as pd
import random
import json
import time
from faker import Faker
from google.colab import files

fake = Faker()
print(" Packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.5 MB/s eta 0:00:00
 Packages installed!


In [2]:
print("="*60)
print("CRAWLING DATA.GOV - WITH CATEGORY FROM THEME & EMAIL EXTRACTION")
print("="*60)

import requests
import json
import time
import pandas as pd
from datetime import datetime

datasets = []
organizations = {}
tags = []
resources = []

base_url = "https://catalog.data.gov/api/3/action/package_search"
start = 0
rows = 50
max_datasets = 2000
fetched = 0

while fetched < max_datasets:
    params = {
        'start': start,
        'rows': min(rows, max_datasets - fetched),
        'sort': 'metadata_created desc'
    }

    print(f"Fetching datasets {start} to {start + params['rows']}...")
    response = requests.get(base_url, params=params, timeout=30)
    data = response.json()

    if not data['success']:
        break

    results = data['result']['results']
    if not results:
        break

    for ds in results:
        # Parse metadata extras
        extras = {}
        for e in ds.get('extras', []):
            key = e['key']
            value = e['value']
            extras[key] = value

        # ========== CATEGORY EXTRACTION ==========
        category = extras.get('category', '')
        if not category:
            theme = extras.get('theme', '')
            if isinstance(theme, list):
                category = theme[0] if theme else ''
            else:
                category = theme
        if not category:
            category = extras.get('topic', '')

        # ========== BUREAU CODE EXTRACTION ==========
        bureau_code = ''
        if 'bureauCode' in extras:
            bureau_value = extras['bureauCode']
            try:
                if isinstance(bureau_value, str):
                    bureau_list = json.loads(bureau_value)
                    bureau_code = bureau_list[0] if bureau_list else ''
                else:
                    bureau_code = bureau_value[0] if bureau_value else ''
            except:
                bureau_code = bureau_value

        # ========== PROGRAM CODE EXTRACTION ==========
        program_code = ''
        if 'programCode' in extras:
            prog_value = extras['programCode']
            try:
                if isinstance(prog_value, str):
                    prog_list = json.loads(prog_value)
                    program_code = prog_list[0] if prog_list else ''
                else:
                    program_code = prog_value[0] if prog_value else ''
            except:
                program_code = prog_value

        # ========== HARVEST INFO EXTRACTION ==========
        harvest_object_id = extras.get('harvest_object_id', '')
        harvest_source_title = extras.get('harvest_source_title', '')
        source_hash = extras.get('source_hash', '')
        source_schema_version = extras.get('source_schema_version', '')

        # ========== CONTACT EMAIL EXTRACTION ==========
        org_contact_email = ''
        possible_email_keys = [
            'contact-email', 'contact_email', 'contactEmail', 'email',
            'publisher_email', 'maintainer_email', 'author_email',
            'contact_point', 'contactPoint', 'metadata_contact',
            'responsible_party', 'point_of_contact'
        ]

        for key in possible_email_keys:
            if key in extras and extras[key]:
                org_contact_email = str(extras[key]).strip()
                if '@' in org_contact_email:
                    break
                else:
                    org_contact_email = ''

        if not org_contact_email:
            org = ds.get('organization', {})
            if org.get('contact_email'):
                org_contact_email = org.get('contact_email')

        if not org_contact_email:
            publisher = extras.get('publisher', ds.get('publisher', ''))
            if '@' in str(publisher):
                org_contact_email = publisher

        # ========== ORGANIZATION INFO ==========
        org = ds.get('organization', {})
        org_name = org.get('name', '')

        if org_name and org_name not in organizations:
            organizations[org_name] = {
                'org_name': org_name,
                'org_type': org.get('type', ''),
                'org_description': org.get('description', '')[:1000] if org.get('description') else '',
                'contact_email': org_contact_email
            }
        elif org_name in organizations and org_contact_email and not organizations[org_name]['contact_email']:
            organizations[org_name]['contact_email'] = org_contact_email

        # ========== DATASET RECORD  ==========
        datasets.append({
            'identifier': ds.get('id', ''),
            'name': ds.get('title', ds.get('name', ''))[:500],
            'description': ds.get('notes', '')[:5000] if ds.get('notes') else '',
            'access_level': extras.get('accessLevel', ds.get('access_level', 'public')),
            'metadata_currentdate': ds.get('metadata_created', ''),
            'updatedate': ds.get('metadata_modified', ''),
            'publisher': extras.get('publisher', ds.get('publisher', '')),
            'maintainer': ds.get('maintainer', ''),
            'bureau_code': bureau_code,
            'harvest_object_id': harvest_object_id,
            'harvest_source_title': harvest_source_title,
            'source_hash': source_hash,
            'source_schema_version': source_schema_version,
            'first_published': extras.get('first_published', ds.get('metadata_created', '')),
            'last_modified': extras.get('last_modified', ds.get('metadata_modified', '')),
            'category': category,
            'program_code': program_code,
            'Source_Datajson_Identifier': extras.get('source_datajson_identifier', ''),
            'org_name': org_name
        })

        # ========== TAGS ==========
        for tag in ds.get('tags', []):
            tag_name = tag.get('display_name', tag.get('name', ''))
            if tag_name:
                tags.append({
                    'identifier': ds.get('id', ''),
                    'tag_name': tag_name[:100]
                })

        # ========== RESOURCES ==========
        for res in ds.get('resources', []):
            resource_url = res.get('url', '')
            resource_format = res.get('format', 'unknown').lower()

            if not resource_format or resource_format == 'unknown':
                if 'github.com' in resource_url:
                    resource_format = 'github'
                elif 'doi.org' in resource_url:
                    resource_format = 'doi'
                elif resource_url.endswith('.xlsx'):
                    resource_format = 'xlsx'
                elif resource_url.endswith('.csv'):
                    resource_format = 'csv'
                elif resource_url.endswith('.pdf'):
                    resource_format = 'pdf'
                elif resource_url.endswith('.docx'):
                    resource_format = 'docx'
                elif resource_url.endswith('.zip'):
                    resource_format = 'zip'
                elif resource_url.endswith('.json'):
                    resource_format = 'json'
                else:
                    resource_format = 'webpage'

            if resource_url:
                resources.append({
                    'identifier': ds.get('id', ''),
                    'source_format': resource_format[:50],
                    'source_url': resource_url[:500]
                })

        fetched += 1
        if fetched % 100 == 0:
            print(f" Processed {fetched} datasets...")

    start += len(results)
    time.sleep(0.5)

print(f"\n Crawled {len(datasets)} REAL datasets!")

# Create DataFrames
df_datasets = pd.DataFrame(datasets)
df_orgs = pd.DataFrame(list(organizations.values()))
df_tags = pd.DataFrame(tags)
df_resources = pd.DataFrame(resources)



CRAWLING DATA.GOV - WITH CATEGORY FROM THEME & EMAIL EXTRACTION
Fetching datasets 0 to 50...
Fetching datasets 50 to 100...
 Processed 100 datasets...
Fetching datasets 100 to 150...
Fetching datasets 150 to 200...
 Processed 200 datasets...
Fetching datasets 200 to 250...
Fetching datasets 250 to 300...
 Processed 300 datasets...
Fetching datasets 300 to 350...
Fetching datasets 350 to 400...
 Processed 400 datasets...
Fetching datasets 400 to 450...
Fetching datasets 450 to 500...
 Processed 500 datasets...
Fetching datasets 500 to 550...
Fetching datasets 550 to 600...
 Processed 600 datasets...
Fetching datasets 600 to 650...
Fetching datasets 650 to 700...
 Processed 700 datasets...
Fetching datasets 700 to 750...
Fetching datasets 750 to 800...
 Processed 800 datasets...
Fetching datasets 800 to 850...
Fetching datasets 850 to 900...
 Processed 900 datasets...
Fetching datasets 900 to 950...
Fetching datasets 950 to 1000...
 Processed 1000 datasets...
Fetching datasets 1000 to 10

In [3]:
print("\n" + "="*60)
print("GENERATING 500 USERS WITH FAKER")
print("="*60)

users = []
for _ in range(500):
    users.append({
        "email": fake.unique.email(),
        "name": fake.name(),
        "address": fake.address().replace("\n", ", ")[:200],
        "birthdate": fake.date_of_birth(minimum_age=18, maximum_age=80).strftime('%Y-%m-%d'),
        "gender": random.choice(["Male", "Female", "Non-binary"]),
        "country": fake.country()
    })

df_users = pd.DataFrame(users)
df_users.to_csv("user.csv", index=False)
print(f" Generated {len(users)} users!")


GENERATING 500 USERS WITH FAKER
 Generated 500 users!


In [4]:
print("\n" + "="*60)
print("GENERATING 500 USER RECORDS")
print("="*60)

categories = ["analytics", "machine learning", "field research"]

project_templates = [
    "Analysis of {topic} using Government Data",
    "ML Model for {topic} Prediction",
    "Research Study: {topic} Trends",
    "{topic} Analytics Project",
    "Predictive Modeling for {topic}",
    "Field Research on {topic}"
]

topics = [
    "Climate Change", "Public Health", "Economic Trends", "Urban Development",
    "Transportation Safety", "Education Outcomes", "Crime Statistics",
    "Housing Market", "Energy Consumption", "Water Quality", "Air Pollution"
]

user_records = []
used_combinations = set()

for _ in range(500):
    user = df_users.sample(1).iloc[0]
    dataset = df_datasets.sample(1).iloc[0]

    topic = random.choice(topics)
    template = random.choice(project_templates)
    project_name = f"{template.format(topic=topic)} (ID: {random.randint(100, 999)})"

    usage_date = fake.date_between(start_date='-365d', end_date='today')

    record = {
        "email": user["email"],
        "identifier": dataset["identifier"],
        "project_name": project_name,
        "project_category": random.choice(categories),
        "usage_date": usage_date.strftime('%Y-%m-%d %H:%M:%S')
    }

    key = (record['email'], record['identifier'], record['project_name'])
    if key not in used_combinations:
        used_combinations.add(key)
        user_records.append(record)

df_user_records = pd.DataFrame(user_records)
df_user_records.to_csv("user_records.csv", index=False)

print(f" Generated {len(df_user_records)} user records!")


GENERATING 500 USER RECORDS
 Generated 500 user records!


In [5]:
df_datasets.to_csv("datasets.csv", index=False)
df_tags.to_csv("tags.csv", index=False)
df_resources.to_csv("resources.csv", index=False)
df_orgs.to_csv("organizations.csv", index=False)
df_users.to_csv("users.csv", index=False)
df_user_records.to_csv("user_records.csv", index=False)